# Toy insect Merkle KG + small→large held-out tests (replayable)

**These are toys, not replicas**: parametric stand-ins sized from database stats, used for handling, sieve and vision tests.
Each toy is a Merkle tree, primitive → part → tagma → **insect hash** (deterministic) → specimen → family → group → collection.
Edge negatives (suspect sizes, below-database sizes) must not be triaged *known*.

Swap the data: point `MANIFEST` at another `manifest.csv` (same columns) or change `printmodels.CA_FAMILIES` and regenerate.

In [1]:
import os, sys
from pathlib import Path
if Path.cwd().name == 'notebooks': os.chdir('..')
sys.path.insert(0, os.getcwd())
import json, csv, collections
from pathlib import Path
from baebugs import printmodels, insect_kg, insect_lomo
MANIFEST = 'print/fuse1/manifest.csv'   # swap here
KG = 'print/fuse1/kg'
rows = list(csv.DictReader(open(MANIFEST)))
print(len(rows), 'toys'); print(collections.Counter(r['training_role'] for r in rows))

91 toys
Counter({'positive': 58, 'edge_negative:family_stats_suspect': 18, 'edge_negative:below_database': 11, 'edge_negative:size_suspect': 4})


## 1. One insect as a Merkle tree, bounding boxes growing level by level

In [2]:
nodes, root = insect_kg.insect_tree(13.5, 'bee')
for n in nodes:
    if n['level'] >= 1:
        print(f"L{n['level']} {n['name']:13s} ext={insect_kg._ext(n['bbox'])} vol={n['volume_mm3']:8.1f} ok={all(n['checks'].values())} {n['hash'][:12]}")
print('insect hash', root['hash'])

L1 head_body     ext=[2.79, 3.7, 4.28] vol=    23.0 ok=True 6f6c265637aa
L1 thorax_body   ext=[4.76, 5.1, 4.9] vol=    61.7 ok=True 4e3e13ca9508
L1 abdomen_body  ext=[7.09, 5.66, 5.1] vol=   106.5 ok=True c79c3a971fff
L1 leg_1L        ext=[3.02, 3.63, 2.59] vol=     2.3 ok=True b114347ca783
L1 leg_1R        ext=[3.02, 3.63, 2.59] vol=     2.3 ok=True beb42cad916a
L1 leg_2L        ext=[0.8, 3.63, 2.59] vol=     1.9 ok=True 15c6a316a3f0
L1 leg_2R        ext=[0.8, 3.63, 2.59] vol=     1.9 ok=True 796fd7cb4802
L1 leg_3L        ext=[5.05, 3.63, 2.59] vol=     2.9 ok=True 67074342995c
L1 leg_3R        ext=[5.05, 3.63, 2.59] vol=     2.9 ok=True ac83defc52d2
L1 antenna_L     ext=[4.41, 4.2, 2.33] vol=     2.7 ok=True 34af9de55b7c
L1 antenna_R     ext=[4.41, 4.2, 2.33] vol=     2.7 ok=True 993ade2f2447
L1 wing_L        ext=[10.12, 3.12, 0.8] vol=    25.3 ok=True 95acf959fc34
L1 wing_R        ext=[10.12, 3.12, 0.8] vol=    25.3 ok=True b664ec3e8dcd
L2 head          ext=[6.38, 7.6, 4.84] vol=   

## 2. Deterministic check, tamper evidence, break point

In [3]:
print('rebuild == root:', insect_kg.rebuild_hash(13.5, 'bee') == root['hash'])
stored = [insect_kg.strip(n) for n in nodes]
stored[4]['bbox'][1][0] += 0.01
print('tampered verify:', insect_kg.verify_tree(stored))
other, _ = insect_kg.insect_tree(13.6, 'bee')
print('13.5 vs 13.6 break point:', insect_kg.break_point(nodes, other)['level'])

rebuild == root: True
tampered verify: (False, 'leg_1L.tibia')
13.5 vs 13.6 break point: 0


## 3. Rebuild the collection KG (writes `kg/`; add `ledger=GitLedger(...)` to commit a break point per level)

In [4]:
index, coll = insect_kg.build_collection(MANIFEST, KG)
print('collection root', coll['hash'][:16]); print(json.load(open(f'{KG}/collection.json'))['level_checks'])
print([ (x['name'], x['failed_checks']) for x in index if not x['all_checks_ok']][:3])

collection root ddbf08d2bf382a2d
{'L0': {'nodes': 1133, 'passed': 1110}, 'L1': {'nodes': 779, 'passed': 779}, 'L2': {'nodes': 273, 'passed': 273}, 'L3': {'nodes': 91, 'passed': 91}}
[('Meloidae_Epicauta_fortis_novel_2.1mm', ['L0:head_body.head_body:fuse_minimum', 'L0:thorax_body.thorax_body:fuse_minimum']), ('thrips_like_fill_1.0mm', ['L0:head_body.head_body:fuse_minimum', 'L0:thorax_body.thorax_body:fuse_minimum']), ('thrips_like_fill_1.5mm', ['L0:head_body.head_body:fuse_minimum', 'L0:thorax_body.thorax_body:fuse_minimum'])]


## 4. Held-out tests: LOMO, LOFO, small→large (s2l), large→small (l2s)
Logic tier only here (deterministic). Add `local='qwen3:1.7b'` (Ollama) and `use_claude=True` to run the cascade.

In [5]:
summary = insect_lomo.run(KG, KG + '/lomo', protocols=('lomo','lofo','s2l','l2s'))
for p, s in summary.items():
    l = s['logic']; print(f"{p:5s} exact {l['exact_hash']}/{l['n']}  triage {l['triage_ok']}/{l['n']}  edge negatives rejected {l['edge_negatives_rejected']}/{l['edge_negatives']}")
    for st, b in s.get('steps', {}).items():
        print('      ', st, '| exact', b['logic']['exact_hash'], '/', b['logic']['n'], '| edge neg rejected', b['logic']['edge_negatives_rejected'], '/', b['logic']['edge_negatives'])

lomo  exact 86/91  triage 88/91  edge negatives rejected 14/33
lofo  exact 86/91  triage 65/91  edge negatives rejected 25/33
s2l   exact 77/78  triage 66/78  edge negatives rejected 4/20
       1: train 1-3mm -> test 3-6mm | exact 9 / 9 | edge neg rejected 0 / 0
       2: train 1-3mm+3-6mm -> test 6-12mm | exact 31 / 32 | edge neg rejected 1 / 5
       3: train 1-3mm+3-6mm+6-12mm -> test 12-20mm | exact 21 / 21 | edge neg rejected 1 / 6
       4: train 1-3mm+3-6mm+6-12mm+12-20mm -> test 20-30mm | exact 16 / 16 | edge neg rejected 2 / 9
l2s   exact 70/75  triage 66/75  edge negatives rejected 12/24
       1: train 20-30mm -> test 12-20mm | exact 21 / 21 | edge neg rejected 0 / 6
       2: train 20-30mm+12-20mm -> test 6-12mm | exact 31 / 32 | edge neg rejected 0 / 5
       3: train 20-30mm+12-20mm+6-12mm -> test 3-6mm | exact 9 / 9 | edge neg rejected 0 / 0
       4: train 20-30mm+12-20mm+6-12mm+3-6mm -> test 1-3mm | exact 9 / 13 | edge neg rejected 12 / 13


## 5. Edge negatives by type

In [6]:
for p in ('lomo','s2l','l2s'):
    d = collections.defaultdict(lambda: [0, 0])
    for l in open(f'{KG}/lomo/{p}.jsonl'):
        r = json.loads(l)
        if r['training_role'].startswith('edge'):
            d[r['training_role']][0] += 1; d[r['training_role']][1] += bool(r['logic']['novel_logic'])
    print(p, {k: f'{v[1]}/{v[0]} rejected' for k, v in d.items()})

lomo {'edge_negative:below_database': '11/11 rejected', 'edge_negative:family_stats_suspect': '0/18 rejected', 'edge_negative:size_suspect': '3/4 rejected'}
s2l {'edge_negative:family_stats_suspect': '2/17 rejected', 'edge_negative:size_suspect': '2/3 rejected'}
l2s {'edge_negative:family_stats_suspect': '0/12 rejected', 'edge_negative:below_database': '11/11 rejected', 'edge_negative:size_suspect': '1/1 rejected'}
